# Benchmark AMT completo — Objetivo específico 1 / Hito H2 (v2)

Evaluación comparativa de arquitecturas de Transcripción Musical Automática sobre
**obras completas** de la partición de prueba de **MAESTRO v3.0.0**, conforme al
Objetivo específico 1 del proyecto:

* ≥ 20 obras completas de la partición de prueba (excluida del entrenamiento original).
* Contraste de al menos tres familias arquitectónicas del estado del arte.
* Umbrales para la arquitectura elegida: **F1_onset ≥ 80 %**, **F1_note (con offset) ≥ 75 %**, **NER ≤ 25 %**.
* Métricas calculadas con `mir_eval` (Raffel et al., 2014).

## Cambios respecto a la v1 (corrida del 2026-08)

1. **Extensión de offsets por pedal en la referencia.** La v1 comparaba contra los
   offsets crudos del MIDI (soltado de tecla). Los modelos entrenados sobre MAESTRO
   aprenden offsets prolongados por el pedal de resonancia (convención de evaluación
   de Onsets and Frames y de Kong et al.), por lo que el F1_note de la v1 colapsaba
   exactamente en el repertorio con pedal denso (Rachmaninoff 3 %, Chopin 10 %) y no
   en el clásico/barroco (Scarlatti 78 %). La v2 prolonga cada offset de la referencia
   mientras el pedal (CC64 ≥ 64) esté pisado, recortando en el siguiente onset de la
   misma altura. El flag `REF_PEDAL_EXTEND` permite reproducir la condición v1.
2. **Caché de notas estimadas crudas.** Cada JSON guarda las notas estimadas, no solo
   las métricas: cambiar la convención de evaluación ya no obliga a re-inferir.
3. **Runner de hFT-Transformer robusto**: el zip del checkpoint trae su propia carpeta
   `checkpoint/`, el `.pkl` se localiza recursivamente, la carga cae a `torch.load`
   si `pickle` falla entre versiones de PyTorch, y los errores imprimen el traceback
   completo.

## Modelos

| Modelo | Familia arquitectónica | Estado |
|---|---|---|
| Kong et al. (2021) — `piano_transcription_inference` | CRNN doble objetivo, alta resolución | Empírico |
| Basic Pitch (Bittner et al., 2022) | CNN ligera multi-tarea | Empírico |
| hFT-Transformer (Toyama et al., 2023) | Transformer jerárquico frecuencia-tiempo | Empírico (checkpoint ISMIR 2023) |
| Onsets and Frames (Hawthorne et al., 2018) | CRNN doble objetivo (original) | Línea base bibliográfica (TF 1.15) |

## Entorno

Kaggle Notebooks, dataset `alonhaviv/the-maestro-dataset-v3-0-0` como *input*, GPU T4,
internet habilitado. Resultados por obra en `/kaggle/working/benchmark_full/` (JSON);
el notebook es **reanudable**.


## 1. Instalación de dependencias

In [ ]:
import subprocess, sys

def pip(*args):
    print(" ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("mir_eval")
pip("--no-deps", "piano_transcription_inference", "torchlibrosa")
pip("--no-deps", "basic-pitch", "tensorflow-io-gcs-filesystem")
pip("--no-deps", "pretty_midi", "resampy", "mido")
pip("librosa", "soundfile")


## 2. Configuración

In [ ]:
import json, os, random, time, glob, traceback
from pathlib import Path

N_WORKS = 20
SEED = 22779              # seleccion determinista de obras
ONSET_TOL = 0.05          # ±50 ms
OFFSET_RATIO = 0.2        # offset: max(50 ms, 20% de la duracion)
REF_PEDAL_EXTEND = True   # prolongar offsets de la referencia con el pedal (CC64)
CACHE_TAG = "v2"          # invalida cache de corridas con otra convencion

TH_F1_ONSET, TH_F1_NOTE, TH_NER = 0.80, 0.75, 0.25   # umbrales del hito H2

DATASET_ROOT = None
for cand in glob.glob("/kaggle/input/*/"):
    hits = glob.glob(cand + "**/maestro-v3.0.0.csv", recursive=True)
    if hits:
        DATASET_ROOT = Path(hits[0]).parent
        break
assert DATASET_ROOT is not None, "Montar el dataset the-maestro-dataset-v3-0-0 como input"
print("Dataset:", DATASET_ROOT)

RESULTS_DIR = Path("/kaggle/working/benchmark_full")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## 3. Selección de obras

In [ ]:
import csv

with open(DATASET_ROOT / "maestro-v3.0.0.csv", newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if r["split"] == "test"]

print(f"Obras en la particion de prueba: {len(rows)}")
rng = random.Random(SEED)
works = rng.sample(rows, N_WORKS)

def resolve(rel):
    p = DATASET_ROOT / rel
    if p.exists():
        return p
    hits = glob.glob(str(DATASET_ROOT / "**" / Path(rel).name), recursive=True)
    assert hits, f"No se encontro {rel}"
    return Path(hits[0])

total_min = 0.0
for i, w in enumerate(works):
    total_min += float(w["duration"]) / 60
    print(f"{i+1:2d}. [{float(w['duration'])/60:5.1f} min] "
          f"{w['canonical_composer']} — {w['canonical_title'][:60]}")
print(f"\nDuracion total: {total_min:.1f} min")


## 4. Métricas y referencia

* **F1_onset**: onset ±50 ms con altura correcta (`offset_ratio=None`).
* **F1_note**: onset ±50 ms y offset dentro de max(50 ms, 20 % de la duración).
* **NER** = (FN + FP) / N_ref sobre el emparejamiento onset+altura de `mir_eval`
  (una sustitución cuenta como FN + FP).
* **Latencia normalizada** = tiempo de inferencia / duración del audio (hito H4: ≤ 1.5).

La referencia prolonga los offsets con el pedal de resonancia: mientras CC64 ≥ 64 la
nota sigue sonando, así que su offset se extiende hasta el soltado del pedal, recortado
en el siguiente onset de la misma altura. Es la convención de evaluación de la línea
Onsets and Frames / Kong et al. sobre MAESTRO.

In [ ]:
import numpy as np
import pretty_midi
import mir_eval.transcription as mt

def _pedal_intervals(pm):
    """Tramos [pisado, soltado) del pedal de resonancia (CC64 >= 64)."""
    ccs = sorted((cc for inst in pm.instruments for cc in inst.control_changes
                  if cc.number == 64), key=lambda c: c.time)
    intervals, down = [], None
    for cc in ccs:
        if cc.value >= 64 and down is None:
            down = cc.time
        elif cc.value < 64 and down is not None:
            intervals.append((down, cc.time))
            down = None
    if down is not None:
        intervals.append((down, float("inf")))
    return intervals

def _extend_with_pedal(notes, pedal):
    """Prolonga cada offset hasta el soltado del pedal que lo cubre,
    recortando en el siguiente onset de la misma altura."""
    if not pedal:
        return notes
    next_onset = {}
    by_pitch = {}
    for n in notes:
        by_pitch.setdefault(n.pitch, []).append(n)
    for pitch, group in by_pitch.items():
        group.sort(key=lambda n: n.start)
        for a, b in zip(group, group[1:]):
            next_onset[id(a)] = b.start
    out = []
    for n in notes:
        end = n.end
        for s, r in pedal:
            if s <= end < r:
                end = r
                break
        end = min(end, next_onset.get(id(n), float("inf")))
        out.append((n.start, max(end, n.start + 1e-6), n.pitch))
    return [pretty_midi.Note(velocity=64, pitch=p, start=s, end=e) for s, e, p in out]

def load_reference(midi_path, extend_pedal=REF_PEDAL_EXTEND):
    pm = pretty_midi.PrettyMIDI(str(midi_path))
    notes = [n for inst in pm.instruments if not inst.is_drum for n in inst.notes]
    if extend_pedal:
        notes = _extend_with_pedal(notes, _pedal_intervals(pm))
    notes.sort(key=lambda n: (n.start, n.pitch))
    intervals = np.array([[n.start, n.end] for n in notes])
    pitches = np.array([pretty_midi.note_number_to_hz(n.pitch) for n in notes])
    return intervals, pitches

def evaluate(ref_i, ref_p, est_i, est_p):
    if len(est_i) == 0:
        return {"f1_onset": 0.0, "p_onset": 0.0, "r_onset": 0.0,
                "f1_note": 0.0, "ner": 1.0, "n_ref": len(ref_i), "n_est": 0}
    p_on, r_on, f_on, _ = mt.precision_recall_f1_overlap(
        ref_i, ref_p, est_i, est_p, onset_tolerance=ONSET_TOL, offset_ratio=None)
    _, _, f_note, _ = mt.precision_recall_f1_overlap(
        ref_i, ref_p, est_i, est_p, onset_tolerance=ONSET_TOL,
        offset_ratio=OFFSET_RATIO, offset_min_tolerance=0.05)
    matching = mt.match_notes(ref_i, ref_p, est_i, est_p,
                              onset_tolerance=ONSET_TOL, offset_ratio=None)
    tp = len(matching)
    fn, fp = len(ref_i) - tp, len(est_i) - tp
    ner = (fn + fp) / len(ref_i) if len(ref_i) else 0.0
    return {"f1_onset": f_on, "p_onset": p_on, "r_onset": r_on,
            "f1_note": f_note, "ner": ner, "n_ref": len(ref_i), "n_est": len(est_i)}


## 5. Runners

Cada *runner* expone `transcribe(audio_path) -> lista de (onset_s, offset_s, midi)`. El modelo se carga una vez; la carga no cuenta en la latencia por obra.

In [ ]:
class KongRunner:
    """CRNN de alta resolucion (PyTorch, Apache 2.0)."""
    name = "Kong et al. (2021)"

    def __init__(self):
        import torch
        from piano_transcription_inference import PianoTranscription, sample_rate
        self.sample_rate = sample_rate
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = PianoTranscription(device=device)

    def transcribe(self, audio_path):
        import librosa
        audio, _ = librosa.load(str(audio_path), sr=self.sample_rate, mono=True)
        out = self.model.transcribe(audio, "/tmp/_kong_tmp.mid")
        return [(e["onset_time"], e["offset_time"], e["midi_note"])
                for e in out["est_note_events"]]


In [ ]:
class BasicPitchRunner:
    """CNN ligera multi-tarea (TF2, Apache 2.0)."""
    name = "Basic Pitch (2022)"

    def __init__(self):
        from basic_pitch.inference import predict
        from basic_pitch import ICASSP_2022_MODEL_PATH
        self._predict = predict
        self._model_path = ICASSP_2022_MODEL_PATH

    def transcribe(self, audio_path):
        _, midi_data, _ = self._predict(str(audio_path), self._model_path)
        return [(n.start, n.end, n.pitch)
                for inst in midi_data.instruments for n in inst.notes]


In [ ]:
class HFTRunner:
    """Transformer jerarquico frecuencia-tiempo (PyTorch, MIT), checkpoint ISMIR 2023."""
    name = "hFT-Transformer (2023)"
    REPO = "https://github.com/sony/hFT-Transformer.git"
    CKPT = "https://github.com/sony/hFT-Transformer/releases/download/ismir2023/checkpoint.zip"

    def __init__(self):
        import subprocess, sys, torch
        work = Path("/kaggle/working/hFT-Transformer")
        if not work.exists():
            subprocess.run(["git", "clone", "--depth", "1", self.REPO, str(work)], check=True)
        # el zip trae su propia carpeta checkpoint/: extraer en la raiz del repo
        if not list(work.rglob("MAESTRO-V3/*.pkl")):
            subprocess.run(["wget", "-q", self.CKPT, "-O", "/tmp/ckpt.zip"], check=True)
            subprocess.run(["unzip", "-oq", "/tmp/ckpt.zip", "-d", str(work)], check=True)
        model_file = next(work.rglob("MAESTRO-V3/*.pkl"))
        print("checkpoint:", model_file)
        if str(work) not in sys.path:
            sys.path.insert(0, str(work))
        with open(work / "corpus" / "config.json", encoding="utf-8") as f:
            config = json.load(f)
        from model import amt
        self.amt = amt.AMT(config, None, verbose_flag=False)
        self.amt.model = self._load_model(model_file, self.amt.device)

    @staticmethod
    def _load_model(path, device):
        import pickle, torch
        try:
            with open(path, "rb") as f:
                model = pickle.load(f)
        except Exception as exc:
            print(f"pickle fallo ({type(exc).__name__}); intentando torch.load")
            model = torch.load(str(path), map_location=device, weights_only=False)
        return model.to(device).eval()

    def transcribe(self, audio_path):
        feature = self.amt.wav2feature(str(audio_path))
        out = self.amt.transcript(feature, mode="combination")
        onset, offset, mpe, velocity = out[4], out[5], out[6], out[7]
        notes = self.amt.mpe2note(a_onset=onset, a_offset=offset, a_mpe=mpe,
                                  a_velocity=velocity,
                                  thred_onset=0.5, thred_offset=0.5, thred_mpe=0.5)
        return [(n["onset"], n["offset"], n["pitch"]) for n in notes]


## 6. Ejecución

Cada JSON guarda métricas **y** las notas estimadas crudas. Las combinaciones ya calculadas se omiten al re-ejecutar; los errores imprimen el traceback completo para diagnóstico.

In [ ]:
RUNNERS = [KongRunner, BasicPitchRunner, HFTRunner]

def est_arrays(est_notes):
    intervals = np.array([[o, f] for o, f, _ in est_notes])
    pitches = np.array([pretty_midi.note_number_to_hz(int(p)) for _, _, p in est_notes])
    return intervals, pitches

def run_benchmark():
    for runner_cls in RUNNERS:
        pending = [w for w in works if not (
            RESULTS_DIR / f"{CACHE_TAG}__{runner_cls.__name__}__{Path(w['midi_filename']).stem}.json"
        ).exists()]
        if not pending:
            print(f"[{runner_cls.name}] completo (cache)")
            continue
        try:
            runner = runner_cls()
        except Exception:
            print(f"[{runner_cls.name}] NO EVALUADO:")
            traceback.print_exc()
            continue
        for w in pending:
            stem = Path(w["midi_filename"]).stem
            out_file = RESULTS_DIR / f"{CACHE_TAG}__{runner_cls.__name__}__{stem}.json"
            try:
                t0 = time.time()
                est_notes = [(round(o, 4), round(f, 4), int(p))
                             for o, f, p in runner.transcribe(resolve(w["audio_filename"]))]
                latency = time.time() - t0
                ref_i, ref_p = load_reference(resolve(w["midi_filename"]))
                m = evaluate(ref_i, ref_p, *est_arrays(est_notes))
                m.update({"model": runner.name, "work": stem,
                          "composer": w["canonical_composer"],
                          "title": w["canonical_title"],
                          "duration_s": float(w["duration"]),
                          "latency_s": latency,
                          "latency_norm": latency / float(w["duration"]),
                          "midi_filename": w["midi_filename"],
                          "est_notes": est_notes})
                out_file.write_text(json.dumps(m))
                print(f"[{runner.name}] {stem[:40]:40s} "
                      f"F1on={m['f1_onset']:.3f} F1note={m['f1_note']:.3f} "
                      f"NER={m['ner']:.3f} lat={m['latency_norm']:.2f}x")
            except Exception:
                print(f"[{runner.name}] {stem}: ERROR")
                traceback.print_exc()

run_benchmark()


### 6b. Recalcular métricas desde el caché (opcional)

Si cambia la convención de evaluación (p. ej. `REF_PEDAL_EXTEND`), esta celda recalcula todas las métricas desde las notas estimadas guardadas, sin re-inferir.

In [ ]:
def recompute_all():
    for p in RESULTS_DIR.glob(f"{CACHE_TAG}__*.json"):
        m = json.loads(p.read_text())
        if "est_notes" not in m or "midi_filename" not in m:
            continue
        ref_i, ref_p = load_reference(resolve(m["midi_filename"]))
        est_notes = [tuple(n) for n in m["est_notes"]]
        m.update(evaluate(ref_i, ref_p, *est_arrays(est_notes)))
        p.write_text(json.dumps(m))
    print("metricas recalculadas")

# recompute_all()


## 7. Agregación y verificación de umbrales

In [ ]:
import pandas as pd

records = []
for p in RESULTS_DIR.glob(f"{CACHE_TAG}__*.json"):
    m = json.loads(p.read_text())
    m.pop("est_notes", None)
    records.append(m)
df = pd.DataFrame(records)
assert not df.empty, "Sin resultados: revisar la celda 6"

agg = df.groupby("model").agg(
    obras=("work", "count"),
    f1_onset=("f1_onset", "mean"), f1_onset_std=("f1_onset", "std"),
    f1_note=("f1_note", "mean"), f1_note_std=("f1_note", "std"),
    ner=("ner", "mean"), ner_std=("ner", "std"),
    lat_norm=("latency_norm", "mean"), lat_norm_p90=("latency_norm", lambda s: s.quantile(0.9)),
).round(4)
display(agg)

print("\nVerificacion de umbrales del Objetivo especifico 1 (hito H2):")
for model, r in agg.iterrows():
    ok = (r.f1_onset >= TH_F1_ONSET and r.f1_note >= TH_F1_NOTE and r.ner <= TH_NER)
    print(f"  {'CUMPLE  ' if ok else 'NO cumple'} {model}: "
          f"F1_onset={r.f1_onset:.2%} (≥{TH_F1_ONSET:.0%}), "
          f"F1_note={r.f1_note:.2%} (≥{TH_F1_NOTE:.0%}), "
          f"NER={r.ner:.2%} (≤{TH_NER:.0%})")

df.to_csv("/kaggle/working/benchmark_full_por_obra.csv", index=False)
agg.to_csv("/kaggle/working/benchmark_full_agregado.csv")


## 8. Tabla final (incluye línea base bibliográfica)

In [ ]:
lines = [
    "| Modelo | Obras | F1_onset | F1_note (con offset) | NER | Latencia norm. (media / p90) |",
    "|---|:---:|:---:|:---:|:---:|:---:|",
]
for model, r in agg.iterrows():
    lines.append(
        f"| {model} | {int(r.obras)} | {r.f1_onset:.2%} ± {r.f1_onset_std:.2%} "
        f"| {r.f1_note:.2%} ± {r.f1_note_std:.2%} | {r.ner:.2%} "
        f"| {r.lat_norm:.3f} / {r.lat_norm_p90:.3f} |")
lines.append(
    "| Onsets and Frames (2018) — linea base bibliografica | — | 94.80% (publicado) "
    "| 82.60% (publicado) | — | — |")
tabla = chr(10).join(lines)
print(tabla)
Path("/kaggle/working/benchmark_full_tabla.md").write_text(tabla)


## 9. Notas de trazabilidad

* Selección determinista (semilla 22779); la lista exacta queda en
  `benchmark_full_por_obra.csv`.
* Los JSON con prefijo `v2__` incluyen las notas estimadas: conservarlos permite
  auditar y recalcular sin GPU.
* Al terminar, descargar los CSV y la tabla Markdown y actualizar
  `docs/literature_review_amt.md` §§6–8.